# Actividad: Multicolinealidad — cuando dos variables cuentan la misma historia

Van a trabajar con un modelo de **regresión lineal múltiple** (el dataset de diabetes de sklearn: 10 variables clínicas para predecir qué tan avanzada está la enfermedad de un paciente un año después).

Objetivo: entender qué le pasa a un modelo cuando dos de sus variables están correlacionadas entre sí (no con el resultado que quieres predecir, sino entre ellas mismas). A esto se le llama **multicolinealidad**.

Es un problema diferente al de "una variable inútil que el modelo usa por casualidad", aquí las dos variables pueden ser genuinamente relevantes, el problema es que se sobreponen entre sí.

## Diccionario de variables

| Variable | Significado |
|---|---|
| `age` | Edad del paciente |
| `sex` | Sexo del paciente |
| `bmi` | Índice de masa corporal |
| `bp` | Presión arterial promedio |
| `s1` | Colesterol total en sangre |
| `s2` | LDL (colesterol "malo") |
| `s3` | HDL (colesterol "bueno") |
| `s4` | Razón colesterol total / HDL |
| `s5` | Posiblemente log de triglicéridos |
| `s6` | Nivel de glucosa en sangre |

Fíjense en `s1` y `s2`: ambas son, en el fondo, formas de medir colesterol. Es razonable sospechar que van a estar relacionadas entre sí.

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

diabetes = load_diabetes(as_frame=True)
df = diabetes.frame
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Tamaño de entrenamiento:", X_train.shape)
print("Tamaño de prueba:       ", X_test.shape)

## Parte A — El modelo completo: miren los coeficientes

Ajusten una regresión lineal con las 10 variables y miren sus coeficientes. En una regresión lineal, el coeficiente de una variable te dice, en teoría, "cuánto cambia la predicción por cada unidad que sube esa variable, dejando las demás fijas".

**Antes de correr la celda:** ¿qué esperarían? Probablemente que `s1` (colesterol total) y `s2` (LDL) tengan coeficientes de tamaño razonable y del mismo signo — al final ambas miden "colesterol", y más colesterol debería apuntar en la misma dirección.

In [ ]:
modelo_completo = LinearRegression()
modelo_completo.fit(X_train, y_train)

print("Coeficientes del modelo completo (10 features):")
for feat, coef in zip(X_train.columns, modelo_completo.coef_):
    print(f"  {feat:>4}: {coef:>10.2f}")

y_pred = modelo_completo.predict(X_test)
print(f"\nMSE test: {mean_squared_error(y_test, y_pred):.2f}")
print(f"R^2 test: {r2_score(y_test, y_pred):.4f}")

**¿Qué observan?** Es muy probable que `s1` les haya salido con un coeficiente enorme y negativo, y `s2` con un coeficiente grande pero positivo — a pesar de que ambas miden colesterol y sería razonable esperar que apuntaran en la misma dirección. Esa es la primera señal de multicolinealidad: coeficientes que no tienen sentido intuitivo, con signos opuestos entre variables que "deberían" comportarse parecido.

Confirmen la sospecha con la matriz de correlación:

In [ ]:
correlaciones = X_train.corr().round(3)
correlaciones

`s1` y `s2` están correlacionadas en aproximadamente **0.90** — casi la misma información medida dos veces. (Si tienen curiosidad, busquen también el par `s3`/`s4`: están correlacionadas en aproximadamente **-0.74**, otro caso sospechoso — regresen a esa pareja en la pregunta de discusión al final.)

Que dos variables estén correlacionadas entre sí no es un error de captura de datos ni un problema del dataset — es información real. El problema es lo que le hace **al modelo**: cuando dos variables cargan casi la misma información, el modelo tiene libertad para repartir el "crédito" entre ellas de formas muy distintas y aun así ajustar casi igual de bien a los datos de entrenamiento. Eso es exactamente lo que vamos a demostrar a continuación.

## Parte B — ¿Qué tan estables son esos coeficientes?

Vamos a hacer una prueba de estabilidad: en vez de ajustar el modelo una sola vez, lo vamos a ajustar 200 veces, cada vez sobre una muestra distinta (con reemplazo) del mismo conjunto de entrenamiento — esto se llama **bootstrap**. Si un coeficiente es "real" y estable, no debería cambiar mucho de una muestra a otra. Si es inestable (síntoma de multicolinealidad), va a brincar mucho — incluso puede cambiar de signo.

La función de abajo ya está lista, no la modifiquen — úsenla para comparar.

In [ ]:
# ============================================================
# Herramienta ya lista — no modificar
# ============================================================
def coeficientes_bootstrap(features_a_reportar, n_iter=200, seed=42):
    """Reajusta el MODELO COMPLETO (las 10 features, igual que en la Parte A)
    n_iter veces sobre remuestreos (bootstrap) de X_train/y_train, y regresa
    un diccionario {feature: array de coeficientes} solo con las columnas
    pedidas en `features_a_reportar` (las demás sí se usan para ajustar el
    modelo en cada remuestreo, solo no se reportan)."""
    rng = np.random.RandomState(seed)
    n = len(X_train)
    todas = list(X_train.columns)
    coefs = {f: [] for f in features_a_reportar}
    for _ in range(n_iter):
        idx = rng.randint(0, n, n)
        Xb = X_train.iloc[idx]
        yb = y_train.iloc[idx]
        m = LinearRegression()
        m.fit(Xb, yb)
        for f in features_a_reportar:
            coefs[f].append(m.coef_[todas.index(f)])
    return {f: np.array(v) for f, v in coefs.items()}

def resumen_coeficientes(coefs_dict):
    for f, arr in coefs_dict.items():
        print(f"  {f:>4}: media={arr.mean():>8.1f}   std={arr.std():>7.1f}   min={arr.min():>8.1f}   max={arr.max():>8.1f}")

print("Listo.")

**Tu turno.** Corre `coeficientes_bootstrap` para el par `['s1', 's2']` y por separado para `['bmi']` (una variable que no tiene ningún par correlacionado fuerte). Compara qué tan grande es el `std` (desviación estándar) de cada una — entre más grande, más "brinca" el coeficiente de una muestra a otra.

```python
coefs_s1_s2 = coeficientes_bootstrap(['s1', 's2'])
resumen_coeficientes(coefs_s1_s2)

coefs_bmi = coeficientes_bootstrap(['bmi'])
resumen_coeficientes(coefs_bmi)
```

In [ ]:
# Corre aquí la comparación de coeficientes_bootstrap para ['s1', 's2'] y ['bmi']



**Para anotar:** ¿el coeficiente de `s1` llegó a cambiar de signo entre remuestreos (a veces positivo, a veces negativo)? ¿Y el de `bmi`? Esa diferencia es exactamente lo que hace peligrosa la multicolinealidad: no es que el modelo prediga mal, es que **no puedes confiar en el coeficiente individual** de las variables correlacionadas — depende demasiado de qué muestra exacta le tocó ver.

## Parte C — Detectarlo sin tener que hacer bootstrap cada vez: el VIF

Hacer 200 remuestreos cada vez que quieras revisar tus variables es mucho trabajo. Hay una métrica estándar para detectar multicolinealidad de un solo vistazo: el **VIF** (Variance Inflation Factor, "factor de inflación de la varianza").

La idea del VIF para una variable X: intenta predecir X usando **todas las demás variables**. Si las demás variables pueden predecir a X casi perfectamente (R² alto), es porque X es casi redundante con ellas — y el VIF será alto. La fórmula es:

$$VIF_i = \frac{1}{1 - R^2_i}$$

donde $R^2_i$ es el R² de regresionar la variable $i$ contra todas las demás. Regla de dedo estándar: **VIF > 10 es señal de multicolinealidad preocupante** (algunos usan un umbral más estricto de 5).

La función de abajo ya está lista:

In [ ]:
# ============================================================
# Herramienta ya lista — no modificar
# ============================================================
def calcular_vif(X):
    """Calcula el VIF de cada columna de X (un DataFrame), regresando
    cada columna contra todas las demás."""
    resultados = {}
    for col in X.columns:
        otras = [c for c in X.columns if c != col]
        m = LinearRegression()
        m.fit(X[otras], X[col])
        r2 = m.score(X[otras], X[col])
        vif = 1 / (1 - r2) if r2 < 1 else float('inf')
        resultados[col] = vif
    return resultados

vifs = calcular_vif(X_train)
print("VIF por variable:")
for feat, v in sorted(vifs.items(), key=lambda kv: -kv[1]):
    marca = "  <-- alto" if v > 10 else ""
    print(f"  {feat:>4}: {v:>6.2f}{marca}")

**Para anotar:** ¿coincide la lista de variables con VIF alto con lo que ya sospechaban por la matriz de correlación y por el experimento de bootstrap?

## Parte D — La corrección: ¿qué pasa si quitamos una?

Si `s1` y `s2` cargan casi la misma información, en teoría deberíamos poder quitar una de las dos sin perder casi nada de poder predictivo — y de paso, arreglar la inestabilidad del coeficiente que quede.

Elige una de las dos (`s1` o `s2`) para quitar, ajusta un modelo con las 9 variables restantes, y compara:
1. El MSE de prueba, contra el del modelo completo de la Parte A.
2. El coeficiente de la variable que sí dejaste (¿sigue siendo tan grande e inestable?).
3. Corre `calcular_vif` otra vez sobre las 9 variables restantes — ¿bajó el VIF de la que dejaste?

```python
cols_reducido = [c for c in X_train.columns if c != 's2']  # o quita 's1', tú decides

modelo_reducido = LinearRegression()
modelo_reducido.fit(X_train[cols_reducido], y_train)

y_pred_reducido = modelo_reducido.predict(X_test[cols_reducido])
print("MSE test (reducido):", mean_squared_error(y_test, y_pred_reducido))
print("R^2 test (reducido):", r2_score(y_test, y_pred_reducido))

for feat, coef in zip(cols_reducido, modelo_reducido.coef_):
    print(f"  {feat:>4}: {coef:>10.2f}")
```

In [ ]:
# Ajusta aquí el modelo reducido (quitando 's1' o 's2') y compara



## Para discutir

1. Al quitar una de las dos variables redundantes, ¿el MSE de prueba cambió mucho? ¿Qué te dice eso sobre cuánta información *nueva* aportaba la variable que quitaste?

2. Si tuvieras que explicarle a un médico "qué tanto afecta el colesterol total (`s1`) al avance de la diabetes" usando el coeficiente del modelo, ¿confiarías en el coeficiente del modelo completo (Parte A) o en el del modelo reducido (Parte D)? ¿Por qué?

3. La multicolinealidad no bajó (casi nada) el desempeño del modelo, pero sí volvió los coeficientes inestables. ¿Por qué eso puede ser un problema real, incluso si las predicciones del modelo son igual de buenas?

4. Encuentra el otro par sospechoso en la matriz de correlación (`s3` y `s4`, correlacionados en aproximadamente -0.74). Sin correr código nuevo — solo con lo que aprendiste — ¿qué esperarías ver si hicieras el mismo experimento de bootstrap con ese par?

5. Si un método automático de selección de features (como los que usaste, o hayas usado, en otra actividad) tuviera que elegir entre `s1` y `s2`, ¿debería quedarse con las dos o con solo una? ¿Por qué?